In [1]:
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import colorcet as cc
import random
import pickle
import igraph as ig
import copy
import sys

sys.path.append("../../scripts")
import readwrite

cfg = readwrite.config()

def get_connected_components_and_leiden(gdf):
    rows, cols = [], []
    for i, geom in enumerate(gdf.geometry):
        for j in gdf.sindex.query(geom.buffer(r)):
            if i < j and geom.distance(gdf.geometry[j]) <= r:
                rows.append(i)
                cols.append(j)

    # Build adjacency matrix
    n = len(gdf)
    A = csr_matrix(( [1] * len(rows), (rows, cols) ), shape=(n, n))

    # Symmetrize (undirected graph)
    A = A + A.T

    # Connected components
    n_components, component_labels = connected_components(A, directed=False)

    # Extract the cluster labels for each cell.
    random.seed(0)
    g = ig.Graph(n=n, edges=zip(rows,cols), directed=False)
    partition = g.community_leiden(objective_function='modularity')
    cluster_labels = np.array(partition.membership,dtype=np.int32)

    return component_labels, cluster_labels

def get_colors(gdf, col_key, cmap = cc.cm.glasbey_light.colors):
    map_colors = {}
    i = 0
    for label in (gdf[col_key].unique()):
        map_colors[label] = cmap[i]
        i+=1
        if i == len(cmap):
            i = 0
    colors = [map_colors[v] for v in gdf[col_key]]
    return colors

def selection_fn(trace, points, selector):
    ''' Callback function for interactive plotly lasso point selection '''

    global selected_centroids_df
    selected_indices = points.point_inds
    
    # Create a DataFrame of selected points
    selected_centroids_df = centroids_df.iloc[selected_indices].reset_index(drop=True)
    
    with out:
        out.clear_output(wait=True)
        if selected_centroids_df.empty:
            print("No points selected.")
        else:
            print("Selected points DataFrame head:")
            print(selected_centroids_df.iloc[:1])

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/anndata/__init__.py:44: F

## Params

In [2]:
donor_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'donor_palette.csv',index_col=0)
gene_sets_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'gene_sets_palette.csv',index_col=0)
cell_type_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'cell_type_palette.csv',index_col=0)

gene_sets = (
    pd.Series({
        "Mitotic": ["MKI67", "CDK1", "UBE2C", "CENPF", "CD80", "ORC6", "STMN1", "TUBA1B"],
        "Goblet_TFF3": ["TFF3","STMN1","TUBA1B","CEACAM1","CEACAM6","RORC","DGKA","CEACAM8","NT5E","MIS18BP1","ATM","IL22RA1","NOTCH1","TUBB",],
        "Stress": ["AREG","CCL20","IL1A","CDKN2B","ANXA1","CXCL1","ISG15","CD68","ID1","IL1B",
                            "DGKA","CDKN1A","VSIR","CEACAM1","TFF3","CEACAM6","IRF1","CEACAM8","TUBB",],
        "Mesenchymal": ["FN1","CXCL6","C1S","CXCL5","C1R","NCAM1","CD74",],
        "Inflammatory": ["CXCL1","AREG","CDKN2B","CDKN1A","ANXA1"],
        "Goblet_MUC5AC": ["REG4", "MUC5AC", "DMBT1", "CCL28", "FOS", "IQGAP2", "JCHAIN", "IGKC", "TOX", "FAS", "IL1B", "RORC",],
        "Interferon": ["IDO1","MX1","IFIT3","CXCL11","CXCL10","ISG15","IFIT2","IRF1","STAT1","CXCL9",],
}).explode().reset_index())
gene_sets.columns = ["source", "target"]


xenium_dir = Path(cfg["xenium_processed_dir"])
xenium_count_correction_dir = Path(cfg["xenium_count_correction_dir"])
xenium_std_seurat_analysis_dir = Path(cfg["xenium_std_seurat_analysis_dir"])
xenium_cell_type_annotation_dir = Path(cfg["xenium_cell_type_annotation_dir"])
results_dir = Path(cfg["results_dir"])

# input params
cellcharter_dir = "cellcharter_cohort"
correction_method = "raw"
segmentation = "proseg_expected"
condition = ["CRC_PDO","CRC_PDO_CAF","CRC_PDO_DEV"]
panel = "all"
normalisation = "lognorm"
reference = "GEO_GSE178341"
method = "rctd_class_aware"
level = "Level1"
xenium_levels = ["segmentation", "condition", "panel", "donor", "sample"]
name_malignant = "Epi"

# qc params
min_n_counts = 20

# fixed params
BATCH_KEY = "dataset_id"
SPATIAL_KEY = "spatial"

segmentations_filter = [segmentation]
conditions_filter = condition
panels_filter = [panel] if panel != "all" else None

# Read results

In [3]:
# read samples
xenium_paths, xenium_annot_paths = readwrite.discover_xenium_paths(
    analysis_dir=xenium_std_seurat_analysis_dir,
    data_dir=xenium_dir,
    annotation_dir=xenium_cell_type_annotation_dir,
    correction_dir=xenium_count_correction_dir,
    normalisation=normalisation,
    reference=reference,
    method=method,
    level=level,
    correction_methods_filter=[correction_method],
    segmentations_filter=[segmentation],
    conditions_filter=conditions_filter,
    panels_filter=panels_filter,
)


# set transcripts=True to load individual transcripts positions)
sds = {}
sds["raw"] = readwrite.read_xenium_samples(
    xenium_paths["raw"], anndata=False, 
    cells_boundaries=True,
    pool_mode="thread", max_workers=6)

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/workflow/analysis/xenium/../../scripts/readwrite.py:350: UserWarning: 
                Couldn't load xenium specs file with pixel size. 
                Not applying scale transformations to shapes.
                Please specify xeniumranger_dir or xenium_specs
                
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__

Error processing [Errno 2] No such file or directory: '/work/PRTNR/CHUV/DIR/rgottar1/spatial/data/norkin_organoid/data/xenium/processed/segmentation/proseg/CRC_PDO/hImmune_v1_dapi/169V/output-XETG00059__0003881__169V_not_1JSK_20250505__170804/raw_results/experiment.xenium'


/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/workflow/analysis/xenium/../../scripts/readwrite.py:350: UserWarning: 
                Couldn't load xenium specs file with pixel size. 
                Not applying scale transformations to shapes.
                Please specify xeniumranger_dir or xenium_specs
                
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'cell_polygons', whi

## segment organoids

In [9]:
# Distance threshold
r = 0

# load manual refinement
dict_manual_refinement = pickle.load(open(cfg['xenium_metadata_dir'] + "dict_manual_refinement_new2.pkl", "rb"))
dict_manual_lasso_refinement = pickle.load(open(cfg['xenium_metadata_dir'] + "dict_manual_lasso_refinement_new2.pkl", "rb"))

for k, sd in sds["raw"].items():
    print(k)

    gdf = sd['cells_boundaries']

    gdf["component"], gdf["cluster"] = get_connected_components_and_leiden(gdf)

    # ensure unique category names across samples by adding sample str
    k_str = "_".join(k)
    gdf["component_str"] = gdf["component"].apply(lambda x: f"{k_str}_comp{x}")
    gdf["cluster_str"]   = gdf["cluster"].apply(lambda x: f"{k_str}_clus{x}")

    # add manual leiden refinement
    idx_manual = gdf['component'].isin(dict_manual_refinement[k])
    gdf["component_and_cluster"] = gdf["component_str"]
    gdf.loc[idx_manual,"component_and_cluster"] = gdf.loc[idx_manual,"cluster_str"]

    # add manual lasso refinement
    gdf["component_and_cluster_and_lasso"] = gdf["component_and_cluster"]

    if k in dict_manual_lasso_refinement:
        # merged lasso + nearest neighbor refinement
        for k_component, dict_lasso in dict_manual_lasso_refinement[k].items():
            comp_mask = gdf["component"] == k_component

            # Track cells covered by any lasso and split by lasso
            lasso_cell_ids = set()
            for k_lasso, idx_lasso in dict_lasso.items():
                idx_mask = gdf["cell_id"].isin(idx_lasso)
                gdf.loc[idx_mask, "component_and_cluster_and_lasso"] = gdf.loc[idx_mask, "component_str"]+f"_lasso{k_lasso}"
                lasso_cell_ids.update(idx_lasso)

            # Find unassigned: in component but not in any lasso
            in_lasso = gdf["cell_id"].isin(lasso_cell_ids)
            unassigned_mask = comp_mask & (~in_lasso)
            if not unassigned_mask.any():
                continue

            # Use lasso cells as nearest neighbor anchors
            assigned_mask = comp_mask & in_lasso

            # Get coordinates (geometry or explicit x/y)
            assigned_coords = np.array(
                [(geom.centroid.x, geom.centroid.y) for geom in gdf.loc[assigned_mask, "geometry"]])
            unassigned_coords = np.array(
                [(geom.centroid.x, geom.centroid.y) for geom in gdf.loc[unassigned_mask, "geometry"]])
 
            # Fit nearest neighbor
            _, nearest_idx = NearestNeighbors(n_neighbors=1).fit(assigned_coords).kneighbors(unassigned_coords)
            nearest = (
                gdf.loc[assigned_mask, "component_and_cluster_and_lasso"]
                .iloc[nearest_idx.flatten()]
                .values
            )

            # Assign nearest labels to unassigned cells
            gdf.loc[unassigned_mask, "component_and_cluster_and_lasso"] = nearest

('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '7_OY6Hsmall', 'output-XETG00059__0033028__7_OY6Hsmall__20250811__161841')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '8_OY6Hsmallmiddle', 'output-XETG00059__0033028__8_OY6Hsmallmiddle__20250811__161841')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '12_OY6Hbighuge', 'output-XETG00059__0033028__12_OY6Hbighuge__20250811__161841')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '10_OY6Hmiddlebig', 'output-XETG00059__0033028__10_OY6Hmiddlebig__20250811__161841')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '1BI7', 'output-XETG00059__0003881__1BI7__20250505__170804')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '1HVQ', 'output-XETG00059__0003381__1HVQ__20250505__170803')
('proseg_expected', 'CRC_PDO_CAF', 'hImmune_v1_dapi', '8samples', 'output-XETG00059__0033131__PDO_8samples__20250821__124602')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '9_11_OY6H_middle_and_big', 'output-XETG00059__0033028__9_11_OY

## manual refinement of assignments

In [ ]:
# # load manual refinement
# dict_manual_refinement = pickle.load(open(cfg['xenium_metadata_dir'] + "dict_manual_refinement_new2.pkl", "rb"))
# dict_manual_lasso_refinement = pickle.load(open(cfg['xenium_metadata_dir'] + "dict_manual_lasso_refinement_new2.pkl", "rb"))

In [ ]:
# # create manual refinement dict
# keys = list(sds["raw"].keys())
# # dict_manual_refinement = {k:[] for k in keys}
# dict_manual_refinement_new = copy.deepcopy(dict_manual_refinement)

# donor_name = '12WP'
# k = [k for k in keys if donor_name in k]
# print(k)
# k = k[0]

# print(dict_manual_refinement[k])

# dict_manual_refinement_new[
#     ('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '18samples', 'output-XETG00059__0033132__PDO_18samples__20250821__124603')
#     ].extend([84,110,140,229,239]) # 123, 186, 212, 237, 351,

# # with open(cfg['xenium_metadata_dir'] + "dict_manual_refinement_new2.pkl", "wb") as f: 
# #     pickle.dump(dict_manual_refinement_new, f)

[('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', '12WP', 'output-XETG00059__0021738__12WP__20250319__172035')]
[5]


In [ ]:
# # plot organoids, create lasso refinement dict with plot_mode='single' 

# # params
# r = 0 # distance threshold
# plot_i = 72 # organoid id to plot for plot_mode == 'single'
# plot_mode = 'single' # plot all organoids at once, all organoids one by one or a single organoid
# k = ('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '18samples', 'output-XETG00059__0033132__PDO_18samples__20250821__124603') # sample name


# # get sample selected sample 
# print(k)
# sd = sds["raw"][k]
# gdf = sd['cells_boundaries']

# gdf["component"], gdf["cluster"] = get_connected_components_and_leiden(gdf)

# gdf["component_and_cluster"] = gdf["component"]
# idx_manual = gdf['component'].isin(dict_manual_refinement[k])
# gdf.loc[idx_manual,"component_and_cluster"] = gdf.loc[idx_manual,"cluster"]

# if plot_mode == 'all_together':
#     # Plot all polygons, colored by cluster_id
#     colors = get_colors(gdf, col_key = "component_and_cluster")
#     fig, ax = plt.subplots(figsize=(10, 10))
#     gdf.plot(color=colors, legend=False, ax=ax,alpha=.5,facecolor='none',aspect=1)
#     ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
#     plt.show()

# elif plot_mode == 'all_separate':
#     gdf_plot = gdf[((gdf['geometry'].centroid.y > 15000) & (gdf['geometry'].centroid.x > 2000) & (gdf['geometry'].centroid.x < 3500))].copy()
#     # gdf_plot = gdf
#     for i in gdf_plot['component'].unique():
#         # if not( i in dict_manual_lasso_refinement[k].keys() or i in dict_manual_refinement[k]):
#         #     continue

#         gdf_ = gdf_plot[gdf_plot['component'] == i]
#         if len(gdf_) < 40 or all(gdf_['cluster'].value_counts()<20):
#             continue

#         fig, ax = plt.subplots(1,2,figsize=(5, 5))
#         fig.suptitle(i,y=.9)
#         gdf_.plot(column="component", cmap=cc.cm.glasbey, legend=False,alpha=.5,facecolor='none',aspect=1,ax=ax[0])

#         gdf_.plot(column="cluster", cmap=cc.cm.glasbey, legend=True,alpha=.5,facecolor='none',aspect=1,ax=ax[1])
        
#         plt.show()

# elif plot_mode == 'single':
#     import plotly.graph_objects as go
#     import ipywidgets

#     try:
#         if 'fig' in locals():
#             fig.close()
#             %xdel fig
#         if 'out' in locals():
#             out.close()
#             # %xdel out
#     except:
#         pass 

#     out = ipywidgets.Output()


#     gdf_ = gdf[gdf['component'] == plot_i]
#     centroids = gdf_.geometry.centroid
#     centroids_df = pd.DataFrame({
#         'cell_id': gdf_.index,
#         'x': centroids.x,
#         'y': centroids.y
#     })



#     # Global variable to store selected points
#     selected_centroids_df = pd.DataFrame(columns=['x', 'y'])

#     # Create a FigureWidget
#     fig = go.FigureWidget(data=go.Scatter(
#         x=centroids_df['x'],
#         y=centroids_df['y'],
#         mode='markers',
#         marker=dict(size=6, color='blue'),
#         # uid=str(uuid.uuid4())  

#     ))

#     # Enable lasso/box selection
#     fig.update_layout(
#         dragmode='lasso',
#         title='Select centroids with lasso or box',
#         width=1000,   # Bigger figure width
#         height=800,   # Bigger figure height
#         xaxis=dict(range=[centroids_df['x'].min(), centroids_df['x'].max()]),
#         yaxis=dict(range=[centroids_df['y'].min(), centroids_df['y'].max()]),
#         xaxis_title='x',
#         yaxis_title='y',
#         showlegend=False
#     )
            
#     # Connect callback
#     fig.data[0].on_selection(selection_fn)

#     # Display
#     display(fig, out)

('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '18samples', 'output-XETG00059__0033132__PDO_18samples__20250821__124603')


ModuleNotFoundError: No module named 'plotly'

In [ ]:
# dict_manual_lasso_refinement[('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', 'OYRI', 'output-XETG00059__0021741__OYRI__20250319__172035')][83] = {
#     0:
#     }

In [ ]:
# print(str(selected_centroids_df['cell_id'].tolist()))

In [ ]:
# with open(cfg['xenium_metadata_dir'] + "dict_manual_lasso_refinement_new2.pkl", "wb") as f:
#     pickle.dump(dict_manual_lasso_refinement, f)

## write results

In [10]:
gdf_all = pd.concat({k:sd['cells_boundaries'] for k, sd in sds["raw"].items()})
gdf_all = gdf_all.reset_index()
gdf_all = gdf_all.rename(columns=dict(zip([f"level_{i}" for i in range(5)],xenium_levels)))
gdf_all["cell_id"] = "proseg-"+gdf_all["cell_id"].astype(str)
gdf_all['full_id'] = gdf_all[xenium_levels+['cell_id']].apply(lambda x: "_".join(x),axis=1)
p=Path(cfg['results_dir'] + "xenium/segment_organoids/organoids_ids.parquet")
p.parent.mkdir(parents=True, exist_ok=True)
gdf_all = gdf_all.set_index('full_id')
gdf_all.to_parquet(p)

## read results

In [153]:
import colorcet as cc
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import sys
from pathlib import Path

sys.path.append("../../scripts")
import readwrite

cfg = readwrite.config()

p=Path(cfg['results_dir'] + "xenium/segment_organoids/organoids_ids.parquet")

gdf = gpd.read_parquet(p)

# filter out small organoids
counts = gdf['component_and_cluster_and_lasso'].value_counts()
idx_keep = counts[counts>20].index
gdf = gdf[gdf['component_and_cluster_and_lasso'].isin(idx_keep)].copy()


# connected components (baseline)
gdf["component_str"]
# leiden clusters
gdf["cluster_str"]
# connected components sub-divided with leiden clusters
gdf["component_and_cluster"]
# connected components sub-divided with leiden clusters and manual lasso
gdf["component_and_cluster_and_lasso"]

plt.style.use('dark_background')
# plot random sample
for sample in gdf['sample'].unique():
    print(sample)
    
    gdf_ = gdf[gdf['sample'] == sample]
    colors = get_colors(gdf_, col_key = "component_and_cluster_and_lasso")

    fig, ax = plt.subplots(figsize=(20, 20))
    gdf_.plot(color=colors, legend=False, ax=ax,alpha=1,facecolor='none',aspect=1)
    # ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
    p = Path(cfg['figures_dir']+f"xenium/segment_organoids/segment_organoids_sample_{sample}.png")
    p.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(p,bbox_inches='tight')
    plt.close()

output-XETG00059__0033028__7_OY6Hsmall__20250811__161841
output-XETG00059__0033028__8_OY6Hsmallmiddle__20250811__161841
output-XETG00059__0033028__12_OY6Hbighuge__20250811__161841
output-XETG00059__0033028__10_OY6Hmiddlebig__20250811__161841
output-XETG00059__0003881__1BI7__20250505__170804
output-XETG00059__0003381__1HVQ__20250505__170803
output-XETG00059__0033131__PDO_8samples__20250821__124602
output-XETG00059__0033028__9_11_OY6H_middle_and_big__20250811__161841
output-XETG00059__0003381__1CNN__20250505__170803
output-XETG00059__0003381__1J25__20250505__170803
output-XETG00059__0003881__1GVB__20250505__170804
output-XETG00059__0003381__OWJ3__20250505__170803
output-XETG00059__0003881__1GNS__20250505__170804
output-XETG00059__0003381__131N__20250505__170803
output-XETG00059__0003881__12NM__20250505__170804
output-XETG00059__0003881__1FMS__20250505__170803
output-XETG00059__0003881__1CI5__20250505__170804
output-XETG00059__0003881__0RL9_not_OZ84__20250505__170804
output-XETG00059__000